# 00_b2 — Czechia: municipality population by age

Part of the `00_b*` series that replaces the five legacy population tables in the local
`dbis` Postgres (read by `traintran/population.py` / `harmonize_population_lau2.ipynb`) with
newer national-statistics files, each brought to the **same age-band widths** and keyed on
`LAU CODE` from `EU-27-LAU-2025-NUTS-2024.xlsx`.

> **Working notebook — not for publication.** It connects to the local DB only to take the legacy
> table as a baseline; once the harmonised output exists, nothing downstream needs the DB.

| | dataset | location |
|---|---|---|
| legacy (DB) | `public.cz_populatoin_lau2_gm_gjsn` | local `dbis` Postgres |
| new (2025) | ČSÚ DataStat `OBY02ET01` — *Počet obyvatel podle pohlaví a základních věkových skupin k 1. 1., 1. 7. a 31. 12.*, municipalities, 31 Dec 2025, bands 0–14 / 15–64 / 65+ (<https://data.csu.gov.cz/datastat/data/VYBER/OBY02ET01?vSet=4&vSel=1>) — **manual download** | `data/eu/ref-national-statistical-offices/cz/Počet obyvatel podle pohlaví a základních věkových skupin k 1. 1., 1. 7. a 31. 12..csv` |

**Legacy notes.** 5-year bands 0–4 … 60–64, 65+ (plus 85+). 98 exact-duplicate rows (6356 → 6258). `so_id_municipality` is already the LAU code.

In [1]:
import os, sys

# Run from the project root so every relative path ('data/...', 'traintran/...') resolves the
# same whether the notebook is launched from notebooks/ or from the root.
_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    os.chdir(os.path.dirname(_here))
sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

Working directory: E:\notes\api_zssk\project


In [2]:
import os
from pathlib import Path

import pandas as pd
# import sqlalchemy as sa

import traintran  # noqa: F401  (fixes PROJ on import)
from traintran import population as pop
from traintran.config import default_config

COUNTRY = 'CZ'
OUT_DIR = Path('data/eu/ref-national-statistical-offices') / COUNTRY.lower()
OUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = default_config()
# Database comparison disabled. The 2025 population comes from the statistical office's files below; the
# database held only the older 2021 tables used for a comparison that feeds no output. To re-enable it, set
# TRAINTRAN_PG_URL in the environment and uncomment these lines and the cells marked '[disabled: ...]'.
# engine = sa.create_engine(os.environ['TRAINTRAN_PG_URL'])
print('Country   :', COUNTRY)
print('Output dir:', OUT_DIR)
# print('Postgres  :', cfg.pg_url.rsplit('@', 1)[-1])

Country   : CZ
Output dir: data\eu\ref-national-statistical-offices\cz


## 1. Legacy — local DB

The raw table(s) as they sit in `dbis`, geometry columns dropped. A parquet snapshot is written
next to the new source (skip-if-present) so the comparison in §3 can later run without the DB.

In [3]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# GEOM_COLS = {'geom', 'geojson', 'wkb_geometry', 'geometry'}
# LEGACY_TABLES = ['public.cz_populatoin_lau2_gm_gjsn']


# def read_table(qualified):
#     schema, table = qualified.split('.', 1)
#     table = table.strip('"')
#     cols = [c['name'] for c in sa.inspect(engine).get_columns(table, schema=schema)
#             if c['name'].lower() not in GEOM_COLS]
#     sel = ', '.join('"' + c.replace('"', '""') + '"' for c in cols)
#     return pd.read_sql(f'select {sel} from {schema}."{table}"', engine)


# legacy_raw = {}
# for t in LEGACY_TABLES:
#     legacy_raw[t] = df = read_table(t)
#     print(f'{t}: {len(df):,} rows x {df.shape[1]} cols')
#     print('  ', list(df.columns))

In [4]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# for t, df in legacy_raw.items():
#     stem = t.replace('.', '_').replace(chr(34), '')
#     snap = OUT_DIR / f'legacy_db_{stem}.parquet'
#     if snap.exists():
#         print('[exists]', snap)
#     else:
#         df.to_parquet(snap, index=False)
#         print('[wrote] ', snap, f'({len(df):,} rows)')

### 1a. Baseline — the current loader on the legacy table

`pop.LOADERS['CZ']` is exactly what `harmonize_population_lau2.ipynb` runs today; its keyed and
banded result is what the 2025 data has to reproduce in structure (same `lau_code` coverage,
same harmonised bands).

In [5]:
lau = pop.load_workbook_lau(cfg.lau_file, countries=(COUNTRY,))
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# legacy = pop.harmonise_bands(pop.LOADERS[COUNTRY](engine, lau), COUNTRY)
# legacy['country'] = COUNTRY
# legacy['ref_year'] = 2021

# print(pop.match_report({COUNTRY: legacy}, lau).to_string(index=False))
# print()
# print('band_method  :', legacy.band_method.unique().tolist())
# print('band_complete:', int(legacy.band_complete.sum()), '/', len(legacy))
# print('pop_total    :', f'{legacy.pop_total.sum():,.0f}')
# print()
# print(legacy.head().to_string())

## 2. New (2025) — national statistical office

Source: ČSÚ DataStat `OBY02ET01`, *Počet obyvatel podle pohlaví a základních věkových skupin
k 1. 1., 1. 7. a 31. 12.* (population by sex and basic age group on 1 Jan, 1 Jul and 31 Dec),
<https://data.csu.gov.cz/datastat/data/VYBER/OBY02ET01?vSet=4&vSel=1>.

**Manual download.** DataStat builds the export in the browser, so there is no stable file URL.
In the selection set: territory *Obce (okres)* (all municipalities), indicator *31. 12. (koncový
stav)*, sex *Celkem*, year *2025*, age *základní věkové skupiny*; export as CSV and save it unchanged
as `cz/Počet obyvatel podle pohlaví a základních věkových skupin k 1. 1., 1. 7. a 31. 12..csv`. (The `.xlsx` export next to it holds the same data and is not read here.)

Unlike the legacy table this source has **only the basic bands 0–14 / 15–64 / 65+**, not 5-year
bands, which is exactly the common schema of the `00_b*` series.

In [6]:
NEW_FILE = OUT_DIR / 'Počet obyvatel podle pohlaví a základních věkových skupin k 1. 1., 1. 7. a 31. 12..csv'

if NEW_FILE.exists():
    print('[exists]', NEW_FILE, f'({NEW_FILE.stat().st_size / 1e6:.1f} MB)')
else:
    raise FileNotFoundError(
        f'{NEW_FILE} is missing. Download it manually from https://data.csu.gov.cz/datastat/data/VYBER/OBY02ET01?vSet=4&vSel=1 '
        '(see the markdown above for the selection) and save it into ' + str(OUT_DIR))

[exists] data\eu\ref-national-statistical-offices\cz\Počet obyvatel podle pohlaví a základních věkových skupin k 1. 1., 1. 7. a 31. 12..csv (6.0 MB)


### 2a. Raw file

Long format: one row per municipality × age group, with each dimension spread over a label
column and three code columns (`X`, `X.Ciselnik`, `X.Polozka`). The municipality name carries its
district in brackets, `Abertamy (okr. Karlovy Vary)`, because Czech municipality names repeat.

In [7]:
cz_raw = pd.read_csv(NEW_FILE, dtype=str, encoding='utf-8')
print(f'{len(cz_raw):,} rows x {cz_raw.shape[1]} cols')
for c in ['Ukazatel', 'Pohlaví', 'Rok', 'Věkové skupiny (základní)', 'MJ_TEXT']:
    print(f'{c:<26}', cz_raw[c].value_counts().to_dict())
print('municipalities           ', cz_raw['UZ5ALLOKR'].nunique())
cz_raw.head(4)

25,032 rows x 23 cols
Ukazatel                   {'Počet obyvatel k 31. 12. (koncový stav)': 25032}
Pohlaví                    {'Celkem': 25032}
Rok                        {'2025': 25032}
Věkové skupiny (základní)  {'Celkem': 6258, '0 - 14 let': 6258, '15 - 64 let': 6258, '65 a více let': 6258}
MJ_TEXT                    {'osoba': 25032}
municipalities            6258


,Ukazatel,IndicatorType,Pohlaví,POHL2,POHL2.Ciselnik,POHL2.Polozka,Rok,CasRB,CasRB.Ciselnik,CasRB.Polozka,...,UZ5ALLOKR.Polozka,Věkové skupiny (základní),VekSkupZakl,VekSkupZakl.Ciselnik,VekSkupZakl.Polozka,Hodnota,MJ_TEXT,MJ_SYMBOL,MJ_CIS,MJ_KOD
0,Počet obyvatel k 31. 12. (koncový stav),2406K2,Celkem,0,#AGG,0,2025,2025,CAS_R,2025,...,554979,Celkem,VEKC,#AGG,0,817.0,osoba,os.,78,80400
1,Počet obyvatel k 31. 12. (koncový stav),2406K2,Celkem,0,#AGG,0,2025,2025,CAS_R,2025,...,554979,0 - 14 let,VEK014,7700,400000610015000,93.0,osoba,os.,78,80400
2,Počet obyvatel k 31. 12. (koncový stav),2406K2,Celkem,0,#AGG,0,2025,2025,CAS_R,2025,...,554979,15 - 64 let,VEK1564,7700,410015610065000,536.0,osoba,os.,78,80400
3,Počet obyvatel k 31. 12. (koncový stav),2406K2,Celkem,0,#AGG,0,2025,2025,CAS_R,2025,...,554979,65 a více let,VEK65AV,7700,410065799999000,188.0,osoba,os.,78,80400


### 2b. Harmonise — one row per municipality

The file must hold exactly one indicator, sex, year and unit (asserted), so pivoting age groups to
columns loses nothing. Names follow the legacy table (`so_id_municipality`,
`so_name_municipality`, `so_name_district`, `popTotal`, `popAge…`), with the three basic bands
as `popAge0to14`, `popAge15to64`, `popAge65over`. The published total must equal the band sum.

In [8]:
for c, expected in [('Ukazatel', 'Počet obyvatel k 31. 12. (koncový stav)'), ('Pohlaví', 'Celkem'),
                    ('Rok', '2025'), ('MJ_TEXT', 'osoba')]:
    assert cz_raw[c].unique().tolist() == [expected], (c, cz_raw[c].unique())
assert (cz_raw['UZ5ALLOKR'] == cz_raw['UZ5ALLOKR.Polozka']).all()

CZ_BANDS = {'VEKC': 'popTotal', 'VEK014': 'popAge0to14',
            'VEK1564': 'popAge15to64', 'VEK65AV': 'popAge65over'}
assert set(cz_raw['VekSkupZakl']) == set(CZ_BANDS), set(cz_raw['VekSkupZakl'])

key = ['UZ5ALLOKR', 'Obce (okres)']
assert not cz_raw.duplicated(key + ['VekSkupZakl']).any(), 'duplicate municipality x age rows'
cz_new = (cz_raw.assign(Hodnota=pd.to_numeric(cz_raw['Hodnota'], errors='raise'))
                .pivot(index=key, columns='VekSkupZakl', values='Hodnota')
                .rename(columns=CZ_BANDS)[list(CZ_BANDS.values())]
                .reset_index().rename_axis(columns=None))

nm = cz_new.pop('Obce (okres)').str.extract(r'^(?P<so_name_municipality>.*?) \(okr\. (?P<so_name_district>.*)\)$')
assert nm.notna().all().all(), 'name without "(okr. …)"'
cz_new.insert(0, 'so_id_municipality', cz_new.pop('UZ5ALLOKR'))
cz_new.insert(1, 'so_name_municipality', nm['so_name_municipality'])
cz_new.insert(2, 'so_name_district', nm['so_name_district'])
cz_new = cz_new.astype({c: 'int64' for c in CZ_BANDS.values()})
cz_new['ref_year'] = 2025
cz_new['ref_date'] = '2025-12-31'

assert cz_new['so_id_municipality'].is_unique
band_sum = cz_new[['popAge0to14', 'popAge15to64', 'popAge65over']].sum(axis=1)
print(f'cz_new: {len(cz_new):,} rows x {cz_new.shape[1]} cols, ids unique')
print('bands sum to popTotal:', f'{(band_sum == cz_new.popTotal).sum():,} / {len(cz_new):,}')
print(f'population total     : {cz_new.popTotal.sum():,}')
print('nulls                :', int(cz_new.isna().sum().sum()))
cz_new.head()

cz_new: 6,258 rows x 9 cols, ids unique
bands sum to popTotal: 6,258 / 6,258
population total     : 10,915,839
nulls                : 0


,so_id_municipality,so_name_municipality,so_name_district,popTotal,popAge0to14,popAge15to64,popAge65over,ref_year,ref_date
0,500011,Želechovice nad Dřevnicí,Zlín,1897,259,1178,460,2025,2025-12-31
1,500020,Petrov nad Desnou,Šumperk,1226,179,752,295,2025,2025-12-31
2,500046,Libhošť,Nový Jičín,1769,302,1085,382,2025,2025-12-31
3,500062,Krhová,Vsetín,2165,339,1390,436,2025,2025-12-31
4,500071,Poličná,Vsetín,1798,269,1137,392,2025,2025-12-31


### 2c. Joinability — legacy `so_id_municipality` vs the new municipality code

Both are ČSÚ municipality codes (*kód obce*), which are also the LAU codes. First the raw id sets,
then the legacy key rule (`pop.cz_key`) against the 2025 workbook — the same report as §1a.

In [9]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# old = (legacy_raw['public.cz_populatoin_lau2_gm_gjsn']
#        [['so_id_municipality', 'so_name_municipality', 'popTotal']]
#        .assign(so_id_municipality=lambda x: [pop.cz_key(v) for v in x.so_id_municipality])
#        .drop_duplicates('so_id_municipality'))
# new = cz_new[['so_id_municipality', 'so_name_municipality', 'so_name_district', 'popTotal']]

# j = old.merge(new, on='so_id_municipality', how='outer', suffixes=('_old', '_new'), indicator=True)
# print(j['_merge'].value_counts().to_string())
# print('\nonly in legacy:')
# print(j.loc[j._merge == 'left_only', ['so_id_municipality', 'so_name_municipality_old', 'popTotal_old']]
#       .to_string(index=False) or '  none')
# print('\nonly in new:')
# print(j.loc[j._merge == 'right_only', ['so_id_municipality', 'so_name_municipality_new', 'popTotal_new']]
#       .to_string(index=False) or '  none')

# both = j[j._merge == 'both']
# renamed = both[both.so_name_municipality_old.str.strip() != both.so_name_municipality_new.str.strip()]
# print(f'\nsame id, different name: {len(renamed)}')
# if len(renamed):
#     print(renamed[['so_id_municipality', 'so_name_municipality_old', 'so_name_municipality_new']]
#           .head(20).to_string(index=False))

In [10]:
cz_new_lau = cz_new.assign(lau_code=[pop.cz_key(v) for v in cz_new.so_id_municipality])
print(pop.match_report({COUNTRY: cz_new_lau}, lau).to_string(index=False))

wb = set(lau['lau_code'])
print('\nnew LAU codes not in the 2025 workbook:')
print(cz_new_lau.loc[~cz_new_lau.lau_code.isin(wb), ['lau_code', 'so_name_municipality', 'popTotal']]
      .to_string(index=False) or '  none')
print('\nworkbook LAU codes not covered by the new file:')
print(lau.loc[~lau.lau_code.isin(cz_new_lau.lau_code), ['lau_code', 'lau_name_national']]
      .to_string(index=False) or '  none')

ctry = cz_new.popTotal.sum()
print(f'\nsum over LAU-mapped rows {cz_new_lau.popTotal.sum():,} vs file total {ctry:,}')

country  source_rows  workbook_lau  matched  source_unmatched  lau_uncovered
     CZ         6258          6258     6258                 0              0

new LAU codes not in the 2025 workbook:


Empty DataFrame
Columns: [lau_code, so_name_municipality, popTotal]
Index: []

workbook LAU codes not covered by the new file:
Empty DataFrame
Columns: [lau_code, lau_name_national]
Index: []

sum over LAU-mapped rows 10,915,839 vs file total 10,915,839


In [11]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# # Same LAU, two vintages: how much did the population move?
# cmp = (legacy[['lau_code', 'pop_total', 'y_0_14', 'y_15_64', 'y_65plus']]
#        .merge(cz_new_lau[['lau_code', 'so_name_municipality', 'popTotal',
#                           'popAge0to14', 'popAge15to64', 'popAge65over']],
#               on='lau_code', how='outer', indicator=True))
# print(cmp['_merge'].value_counts().to_string())
# b = cmp[cmp._merge == 'both'].copy()
# b['diff_%'] = 100 * (b.popTotal - b.pop_total) / b.pop_total
# print(f'\ntotal  legacy {b.pop_total.sum():,.0f}  ->  new {b.popTotal.sum():,.0f}')
# print(b['diff_%'].describe().round(2).to_string())
# print('\nband shares (%), legacy -> new:')
# for o, n in [('y_0_14', 'popAge0to14'), ('y_15_64', 'popAge15to64'), ('y_65plus', 'popAge65over')]:
#     print(f'  {o:<9} {100 * b[o].sum() / b.pop_total.sum():5.1f} -> {100 * b[n].sum() / b.popTotal.sum():5.1f}')
# print('\nlargest relative changes:')
# print(b.reindex(b['diff_%'].abs().sort_values(ascending=False).index)
#        .head(10)[['lau_code', 'so_name_municipality', 'pop_total', 'popTotal', 'diff_%']]
#        .round(1).to_string(index=False))

### 2d. Territorial changes

Same mechanism as Austria: a merged municipality stays one row under its new code and
`replaces_lau` lists the workbook codes it covers. Czechia needs no entry for now (the code
sets match one to one); the assert keeps it that way if ČSÚ re-issues the file after a merger.

In [12]:
# new LAU code -> workbook (2025) LAU codes it replaces
CZ_MERGERS = {}

cz_new_lau['replaces_lau'] = cz_new_lau['lau_code'].map(lambda c: ';'.join(CZ_MERGERS.get(c, [])) or None)

covered = set(cz_new_lau['lau_code']) | {o for olds in CZ_MERGERS.values() for o in olds}
unexplained_new = sorted(set(cz_new_lau['lau_code']) - wb - set(CZ_MERGERS))
unexplained_wb = sorted(wb - covered)
assert not unexplained_new and not unexplained_wb, (unexplained_new, unexplained_wb)
print(f'new rows            : {len(cz_new_lau):,}')
print(f'workbook LAU covered: {len(wb & covered):,} / {len(wb):,}  ({len(CZ_MERGERS)} mergers)')

new rows            : 6,258
workbook LAU covered: 6,258 / 6,258  (0 mergers)


### 2e. Join to the v4atua LAU subset (GeoJSON)

Only the Czech features of the LAU 2024 subset (`CNTR_CODE = 'CZ'`), key `GISCO_ID = 'CZ_<LAU code>'`.

In [13]:
import geopandas as gpd

LAU_SUBSET = Path('data/eu/ref-lau-2024-01m/subsetv4atua/LAU_RG_01M_2024_4326.geojson')
cz_geo = gpd.read_file(LAU_SUBSET, where="CNTR_CODE = 'CZ'")
cz_geo['lau_code'] = cz_geo['GISCO_ID'].str.split('_', n=1).str[1]
assert cz_geo['lau_code'].is_unique
print(f'{LAU_SUBSET.name}: {len(cz_geo):,} CZ features')

old_to_new = {o: n for n, olds in CZ_MERGERS.items() for o in olds}
cz_geo['lau_code_2025'] = cz_geo['lau_code'].replace(old_to_new)

jg = cz_geo.drop(columns='geometry').merge(
    cz_new_lau[['lau_code', 'so_name_municipality', 'so_name_district', 'popTotal', 'replaces_lau']],
    left_on='lau_code_2025', right_on='lau_code', how='outer', suffixes=('_geo', ''), indicator=True)
print('\n' + jg['_merge'].value_counts().to_string())
print('\ngeojson only:', jg.loc[jg._merge == 'left_only', 'lau_code_2025'].tolist())
print('new file only:', jg.loc[jg._merge == 'right_only', 'lau_code'].tolist())

m = jg[jg._merge == 'both'].copy()
m['diff_%'] = 100 * (m.popTotal - m.POP_2024) / m.POP_2024
print(f'\npopulation  GISCO POP_2024 {m.POP_2024.sum():,.0f}  vs  new popTotal {m.popTotal.sum():,.0f}')
print(m['diff_%'].describe().round(2).to_string())
name_diff = m[m.LAU_NAME.str.strip() != m.so_name_municipality.str.strip()]
print(f'\nsame code, different name: {len(name_diff)}')
print(name_diff[['lau_code_2025', 'LAU_NAME', 'so_name_municipality']].head(15).to_string(index=False))

LAU_RG_01M_2024_4326.geojson: 6,258 CZ features

_merge
both          6258
left_only        0
right_only       0

geojson only: []
new file only: []

population  GISCO POP_2024 10,900,555  vs  new popTotal 10,915,839
count    6254.00
mean        0.47
std         4.10
min       -39.89
25%        -1.56
50%         0.21
75%         2.30
max        54.38

same code, different name: 0
Empty DataFrame
Columns: [lau_code_2025, LAU_NAME, so_name_municipality]
Index: []


## 3. Write — common-schema dump

One CSV per country, same columns in all five `00_b*` notebooks, read by
`00_extract_base_data_eu.ipynb` §2c to attach the 2025 population to the LAU subset GeoJSON.

| column | content |
|---|---|
| `lau_code`, `gisco_id` | LAU code (text) and `<CC>_<lau_code>`, the subset key |
| `country`, `lau_name` | country code, national name |
| `pop_total`, `y_0_14`, `y_15_64`, `y_65plus` | persons (`pop.HARMONISED_BANDS`) |
| `band_method`, `ref_year`, `ref_date`, `source` | `exact`; reference year/date; dataset id |
| `native_id` | the source's own key |
| `replaces_lau` | workbook LAU codes a merged row covers (`;`-joined), else empty |
| `includes_native` | native units summed into the row (`;`-joined), else empty |

Skips if the file exists; set `REWRITE_DUMP = True` to overwrite. Read it back with `dtype=str`
for the code columns (HU codes have leading zeros).

In [14]:
REWRITE_DUMP = False
DUMP_COLS = ['lau_code', 'gisco_id', 'country', 'lau_name', 'pop_total', *pop.HARMONISED_BANDS,
             'band_method', 'ref_year', 'ref_date', 'source', 'native_id', 'replaces_lau', 'includes_native']
DUMP_FILE = OUT_DIR / f'lau_population_2025_{COUNTRY.lower()}.csv'

f = cz_new_lau.reset_index(drop=True)
dump = pd.DataFrame({
    'lau_code': f['lau_code'].astype(str),
    'gisco_id': COUNTRY + '_' + f['lau_code'].astype(str),
    'country': COUNTRY,
    'lau_name': f['so_name_municipality'],
    'pop_total': f['popTotal'], 'y_0_14': f['popAge0to14'], 'y_15_64': f['popAge15to64'], 'y_65plus': f['popAge65over'],
    'band_method': 'exact',
    'ref_year': f['ref_year'], 'ref_date': f['ref_date'],
    'source': 'CZSO_OBY02ET01',
    'native_id': f['so_id_municipality'].astype(str),
    'replaces_lau': f['replaces_lau'],
    'includes_native': None,
})[DUMP_COLS]

assert dump['lau_code'].is_unique and dump['gisco_id'].is_unique
assert (dump[pop.HARMONISED_BANDS].sum(axis=1) == dump['pop_total']).all(), 'bands do not add up'
assert dump[['pop_total', *pop.HARMONISED_BANDS]].notna().all().all()
replaced = {o for r in dump['replaces_lau'].dropna() for o in r.split(';')}
merged_new = set(dump.loc[dump['replaces_lau'].notna(), 'lau_code'])
assert (set(dump['lau_code']) - merged_new) | replaced == set(lau['lau_code']), 'dump does not cover the workbook LAUs'

if DUMP_FILE.exists() and not REWRITE_DUMP:
    print('[exists]', DUMP_FILE, '-- set REWRITE_DUMP = True to overwrite')
else:
    dump.to_csv(DUMP_FILE, index=False, encoding='utf-8')
    print('[wrote] ', DUMP_FILE)
back = pd.read_csv(DUMP_FILE, dtype={'lau_code': str, 'gisco_id': str, 'native_id': str})
assert list(back.columns) == DUMP_COLS and back['lau_code'].tolist() == dump['lau_code'].tolist(), 'file differs from dump'
print(f'{len(back):,} rows, population {back.pop_total.sum():,}; '
      f"merged rows {int(back.replaces_lau.notna().sum())}, aggregated rows {int(back.includes_native.notna().sum())}")
back.head(3)

[exists] data\eu\ref-national-statistical-offices\cz\lau_population_2025_cz.csv -- set REWRITE_DUMP = True to overwrite
6,258 rows, population 10,915,839; merged rows 0, aggregated rows 0


,lau_code,gisco_id,country,lau_name,pop_total,y_0_14,y_15_64,y_65plus,band_method,ref_year,ref_date,source,native_id,replaces_lau,includes_native
0,500011,CZ_500011,CZ,Želechovice nad Dřevnicí,1897,259,1178,460,exact,2025,2025-12-31,CZSO_OBY02ET01,500011,NaN,NaN
1,500020,CZ_500020,CZ,Petrov nad Desnou,1226,179,752,295,exact,2025,2025-12-31,CZSO_OBY02ET01,500020,NaN,NaN
2,500046,CZ_500046,CZ,Libhošť,1769,302,1085,382,exact,2025,2025-12-31,CZSO_OBY02ET01,500046,NaN,NaN
